# Dijkstra 算法与 Bellman-Ford 算法

在本部分中，你将实现我们在课堂上见过的两种紧密相关的最短路径算法。

## 环境配置说明：
### 如果在本地运行：
#### 首次配置
* 安装Anaconda：访问 https://www.anaconda.com/products/distribution
* 创建conda环境：`conda create -n cs131232 python=3.8`
  * 详见 https://conda.io/projects/conda/en/latest/user-guide/tasks/manage-environments.html
* 激活环境：`conda activate cs131232`
* 安装pip：`conda install pip`
* 安装Jupyter：`conda install jupyter`
#### 每次启动
* 确保激活环境：`conda activate cs131232`
* 启动Jupyter：`jupyter notebook` 或 `jupyter lab`
* 运行以下单元格**并在需要时重启内核**

In [ ]:
# Install dependencies
!pip install -r requirements.txt --quiet

In [ ]:
import otter

assert (otter.__version__ >= "4.4.1"), "Please reinstall the requirements and restart your kernel."

grader = otter.Notebook("shortest-paths.ipynb")
import numpy as np
from numpy.random import randint
from time import time
import heapq
import tqdm
import networkx as nx
import matplotlib.pyplot as plt
import numpy.random as random

rng_seed = 0

### 代码中的图表示法（第二部分！！！）

与上次作业不同，我们的图现在是带权重的，因此除了边的信息外，还需要同时存储权重。使用边列表表示法时，我们可以通过创建元组列表 `(u, v, w)` 来表示权重为 $w$ 的有向边 $(u, v)$。

然而，与上次类似，我们希望用邻接表来表示图。我们可以在 `adj_list[u]` 中存储元组 `(v, w)` 来表示权重为 $w$ 的有向边 $(u, v)$。

In [ ]:
def generate_adj_list(n, edge_list):
    """
    args:
        n:int = number of nodes in the graph. The nodes are labelled with integers 0 through n-1
        edge_list:List[Tuple[int,int,int]] = edge list where each tuple (u,v,w) represents the directed 
            and weighted edge (u,v,w) in the graph
    return:
        A List[List[Tuple[int, int]]] representing the adjacency list 
    """
    adj_list = [[] for i in range(n)] 
    for u, v, w in edge_list:
        adj_list[u].append((v, w))
    for nodes in adj_list:
        nodes.sort()
    return adj_list

def draw_graph(adj_list):
    """Utility method for visualizing graphs

    args:
        adj_list (List[List[Tuple[int, int]]]): adjacency list of the graph given by generate_adj_list
    """
    G = nx.DiGraph()
    for u in range(len(adj_list)):
        for v, w in adj_list[u]:
            G.add_edge(u, v, weight=w)
    nx.draw(G, with_labels=True)

### Python 中的优先队列

为简单起见，我们为你提供了以下优先队列的实现，其底层使用了 `heapq` 模块。我们的实现了课堂中所讲的 `insert` 和 `deleteMin` 操作，但不包含 `decreaseKey` 操作，因为 Python 的 `heapq` 模块不支持它。不过，事实证明，完成本次作业中的任何算法都不需要 `decreaseKey`——我们将在相关部分详细讨论这一点。

你不需要理解我们的实现，但如果你有兴趣了解更多，`heapq` 模块的文档可在此处获取：https://docs.python.org/3/library/heapq.html

你可以根据需要对这一实现进行修改，但如果你这样做，请确保修改后的实现是正确的且高效的。**一个比我们的实现更慢的优先队列实现可能会导致你的代码在评测时超时。**

In [ ]:
class PriorityQueue:
    def __init__(self):
        self.queue = []

    def insert(self, priority, item):
        heapq.heappush(self.queue, (priority, item))

    def deleteMin(self):
        return heapq.heappop(self.queue)[1]

    def __len__(self):
        return len(self.queue)    

The operations `insert` and `deleteMin` work as follows:

In [ ]:
pq = PriorityQueue()

for i in range(10):
    pq.insert(-i, i)

# should print 9, 8, 7
print(pq.deleteMin())
print(pq.deleteMin())
print(pq.deleteMin())

### Q0: `update` 函数（可选）

我们可以实现一个子程序 `update` 来更新图中顶点的距离。Bellman-Ford 算法可以被视为应用一系列 `update` 操作（如课堂中所讲），但事实证明，Dijkstra 算法也可以做到这一点！在本部分中，你可以实现一个 `update` 函数，该函数可在两种算法中使用。

由于后面我们会要求你重构出实际的最短路径，因此在更新边时记录每个顶点的前驱可能会很有用。

如果你愿意，可以安全地跳过此部分，因为这部分不计分。

In [ ]:
def update(u, v, w, dists, prev=None):
    """Updates the distance dists[v] in the dists array based on the
    update procedure described in DPV.

    Args:
        u (int): starting node of th eedge (u, v)
        v (int): ending node of the edge (u, v)
        w (int): weight of the edge (u, v)
        dists (List[int]): The distance array used in our shortest-path algorithm. The source node
            s actually does not need to be specified here, but it will be needed in the 
            shortest-path algorithm.
        prev (List[int]): Array keeping track of the previous node along the shortest path
    """
    if dists[u] + w < dists[v]:
        dists[v] = dists[u] + w
        if prev is not None:
            prev[v] = u

### Q1: Dijkstra 算法

由于我们无法使用 `decreaseKey` 操作，因此做如下修改：不调用 `decreaseKey`，而是直接将更新距离后的顶点新副本插入优先队列，并不移除旧副本。这可能导致同一顶点在优先队列中存在多个副本，但这不会影响算法的正确性。

> **任务 1：** 使用 Dijkstra 算法计算从 $s$ 到 $t$ 的最短路径，并将该路径作为节点列表返回。

例如，路径 ${s \to a \to b \to c \to t}$ 对应列表 `[s, a, b, c, t]`。

所有边的权重均为非负数。如果不存在路径，则返回空列表 `[]`。如果存在多条最短路径，你可以返回其中任意一条。

_分值：_ 25

In [ ]:
def dijkstra(G, s, t):
    """Implements Dijkstra's algorithm to find the shortest weighted path from s to t.

    Args:
        G (List[List[Tuple[int, int]]]): The weighted adjacency list of the graph
        s (int): The source node
        t (int): The target node

    Returns:
        List[int]: A list of nodes starting with s and ending with t representing the
        shortest weighted s to t path if it exists. Returns an empty list otherwise.
    """
    n = len(G)
    INF = float("inf")
    dists = [INF] * n
    prev = [-1] * n
    dists[s] = 0

    # heapq has no decreaseKey, so stale copies are simply ignored when popped.
    pq = [(0, s)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dists[u]:
            continue
        if u == t:
            break
        for v, w in G[u]:
            new_dist = d + w
            if new_dist < dists[v]:
                dists[v] = new_dist
                prev[v] = u
                heapq.heappush(pq, (new_dist, v))

    if dists[t] == INF:
        return []

    path = []
    node = t
    while node != -1:
        path.append(node)
        node = prev[node]
    path.reverse()
    return path

In [ ]:
grader.check("q1")

### Q2: Bellman-Ford 算法

> **任务 2：** 使用 Bellman-Ford 算法计算从 $s$ 到 $t$ 的最短路径，并将该路径作为节点列表返回。

例如，路径 ${s \to a \to b \to c \to t}$ 对应列表 `[s, a, b, c, t]`。

如果不存在 $s \to t$ 路径，或者图中存在负环，则返回空列表 `[]`。如果存在多条最短路径，你可以返回其中任意一条。

_分值：_ 25

In [ ]:
def bellman_ford(G, s, t):
    """Implements the Bellman-Ford algorithm for single-source shortest paths.

    Args:
        G (List[List[Tuple[int, int]]]): The weighted adjacency list of the graph
        s (int): The source node
        t (int): The target node

    Returns:
        List[int]: A list of nodes starting with s and ending with t representing the
        shortest weighted s to t path if it exists. Returns an empty list otherwise.
    """
    n = len(G)
    edges = [(u, v, w) for u in range(n) for v, w in G[u]]

    # The assignment asks for [] whenever the graph contains any negative cycle,
    # even one unreachable from s. Starting every vertex at distance 0 is
    # equivalent to adding a virtual source with 0-weight edges to all vertices.
    dists = [0] * n
    for _ in range(n):
        updated = False
        for u, v, w in edges:
            new_dist = dists[u] + w
            if new_dist < dists[v]:
                dists[v] = new_dist
                updated = True
        if not updated:
            break
    else:
        return []

    INF = float("inf")
    dists = [INF] * n
    prev = [-1] * n
    dists[s] = 0
    for _ in range(n - 1):
        updated = False
        for u, v, w in edges:
            if dists[u] != INF:
                new_dist = dists[u] + w
                if new_dist < dists[v]:
                    dists[v] = new_dist
                    prev[v] = u
                    updated = True
        if not updated:
            break

    if dists[t] == INF:
        return []

    path = []
    node = t
    while node != -1:
        path.append(node)
        node = prev[node]
    path.reverse()
    return path

In [ ]:
grader.check("q2")

## 提交

确保在运行下面的单元格之前，你已经按顺序运行了笔记本中的所有单元格，以便所有图像/图表都能在输出中显示。下面的单元格将为你生成一个用于提交的压缩文件。

In [ ]:
grader.export(pdf=False, force_save=True, run_tests=True)